<a href="https://colab.research.google.com/github/ravindravala/AI/blob/main/toy_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# BLOCK 1 — Setup + Tiny Text Dataset
# ============================================================

import torch
import torch.nn as nn
from torch.nn import functional as F

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Reproducibility
torch.manual_seed(42)

# ------------------------------------------------------------
# A small plain text page.
# You can replace this with any one-page text later.
# ------------------------------------------------------------

text = """
Ravi is learning how artificial intelligence models work.

A language model learns patterns from text. It does not truly understand like a human,
but it can learn to predict what comes next.

If the model sees many examples, it becomes better at continuing text.

Fine tuning means training a model further on a smaller, specific dataset.
Instruction tuning means showing the model examples of instructions and answers.

For example:
Instruction: Say hello.
Answer: Hello, how can I help you?

Instruction: Explain AI simply.
Answer: AI is software that learns patterns from data.

Instruction: What is fine tuning?
Answer: Fine tuning is additional training on a specific task or style.

This is a tiny experiment. The model is very small. It will not be powerful,
but it will help us understand tokenization, training, generation, and instruction tuning.
"""

print("Total characters in text:", len(text))
print("\n--- Text preview ---")
print(text[:500])

Using device: cpu
Total characters in text: 858

--- Text preview ---

Ravi is learning how artificial intelligence models work.

A language model learns patterns from text. It does not truly understand like a human,
but it can learn to predict what comes next.

If the model sees many examples, it becomes better at continuing text.

Fine tuning means training a model further on a smaller, specific dataset.
Instruction tuning means showing the model examples of instructions and answers.

For example:
Instruction: Say hello.
Answer: Hello, how can I help you?

Instr


In [2]:
# ============================================================
# BLOCK 2 — Character Tokenizer
# ============================================================

# Get all unique characters from the text
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Unique characters:")
print(chars)

print("\nVocabulary size:", vocab_size)

# Create mappings:
# character -> integer
# integer -> character
stoi = { ch: i for i, ch in enumerate(chars) }
itos = { i: ch for i, ch in enumerate(chars) }

# Encode: text string -> list of token ids
def encode(s):
    return [stoi[c] for c in s]

# Decode: list of token ids -> text string
def decode(ids):
    return "".join([itos[i] for i in ids])

# Test tokenizer
sample = "Instruction: Say hello."
encoded = encode(sample)
decoded = decode(encoded)

print("\nSample text:")
print(sample)

print("\nEncoded:")
print(encoded)

print("\nDecoded:")
print(decoded)

# Encode full dataset
data = torch.tensor(encode(text), dtype=torch.long)

print("\nFull data tensor shape:", data.shape)
print("First 50 token ids:", data[:50].tolist())
print("Decoded first 50 chars:")
print(decode(data[:50].tolist()))

Unique characters:
['\n', ' ', ',', '.', ':', '?', 'A', 'E', 'F', 'H', 'I', 'R', 'S', 'T', 'W', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

Vocabulary size: 39

Sample text:
Instruction: Say hello.

Encoded:
[10, 27, 31, 32, 30, 33, 17, 32, 23, 28, 27, 4, 1, 12, 15, 37, 1, 22, 19, 25, 25, 28, 3]

Decoded:
Instruction: Say hello.

Full data tensor shape: torch.Size([858])
First 50 token ids: [0, 11, 15, 34, 23, 1, 23, 31, 1, 25, 19, 15, 30, 27, 23, 27, 21, 1, 22, 28, 35, 1, 15, 30, 32, 23, 20, 23, 17, 23, 15, 25, 1, 23, 27, 32, 19, 25, 25, 23, 21, 19, 27, 17, 19, 1, 26, 28, 18, 19]
Decoded first 50 chars:

Ravi is learning how artificial intelligence mode


In [3]:
# ============================================================
# BLOCK 3 — Train/Validation Split + Batch Creation
# ============================================================

# How many characters the model sees at once
block_size = 64

# How many examples in one training batch
batch_size = 16

# Split data: 90% train, 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("Train tokens:", len(train_data))
print("Validation tokens:", len(val_data))

def get_batch(split):
    """
    Returns:
    x: input tokens  shape  [batch_size, block_size]
    y: target tokens shape  [batch_size, block_size]
    """
    source = train_data if split == "train" else val_data

    # Random starting positions
    ix = torch.randint(len(source) - block_size, (batch_size,))

    # x gets block_size characters
    x = torch.stack([source[i:i + block_size] for i in ix])

    # y gets the next character for each x position
    y = torch.stack([source[i + 1:i + block_size + 1] for i in ix])

    return x.to(device), y.to(device)

# Test one batch
xb, yb = get_batch("train")

print("Input batch shape:", xb.shape)
print("Target batch shape:", yb.shape)

print("\n--- First training example ---")
print("INPUT:")
print(decode(xb[0].tolist()))

print("\nTARGET:")
print(decode(yb[0].tolist()))

Train tokens: 772
Validation tokens: 86
Input batch shape: torch.Size([16, 64])
Target batch shape: torch.Size([16, 64])

--- First training example ---
INPUT:
ining on a specific task or style.

This is a tiny experiment. T

TARGET:
ning on a specific task or style.

This is a tiny experiment. Th


In [4]:
# ============================================================
# BLOCK 4 — Tiny Transformer Config + Self-Attention Head
# ============================================================

# Tiny model hyperparameters
n_embd = 64          # embedding size
n_head = 4           # number of attention heads
n_layer = 2          # number of transformer blocks
dropout = 0.1

learning_rate = 1e-3
max_iters = 2000
eval_interval = 200
eval_iters = 50

print("Vocab size:", vocab_size)
print("Block size:", block_size)
print("Embedding size:", n_embd)
print("Attention heads:", n_head)
print("Transformer layers:", n_layer)


class Head(nn.Module):
    """
    One self-attention head.
    """

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Causal mask.
        # This prevents the model from looking at future tokens.
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)    # [B, T, head_size]
        q = self.query(x)  # [B, T, head_size]

        # Attention scores
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)  # [B, T, T]

        # Apply causal mask
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        # Convert scores to probabilities
        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        v = self.value(x)  # [B, T, head_size]

        # Weighted sum of values
        out = wei @ v      # [B, T, head_size]

        return out


# Quick shape test
test_head = Head(head_size=n_embd // n_head).to(device)
test_x = torch.randn(batch_size, block_size, n_embd).to(device)
test_out = test_head(test_x)

print("Self-attention head output shape:", test_out.shape)

Vocab size: 39
Block size: 64
Embedding size: 64
Attention heads: 4
Transformer layers: 2
Self-attention head output shape: torch.Size([16, 64, 16])


In [5]:
# ============================================================
# BLOCK 5 — Multi-Head Attention + FeedForward + Block
# ============================================================

class MultiHeadAttention(nn.Module):
    """
    Multiple self-attention heads running in parallel.
    """

    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size) for _ in range(num_heads)
        ])

        # Project concatenated heads back to embedding size
        self.proj = nn.Linear(num_heads * head_size, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run all heads and concatenate their outputs
        out = torch.cat([h(x) for h in self.heads], dim=-1)

        # Final projection
        out = self.proj(out)

        out = self.dropout(out)

        return out


class FeedForward(nn.Module):
    """
    Simple MLP applied independently to each token.
    """

    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    One Transformer block:
    self-attention + feed-forward, both with residual connections.
    """

    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Pre-norm Transformer style
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x


# Quick shape test
test_block = Block(n_embd, n_head).to(device)
test_x = torch.randn(batch_size, block_size, n_embd).to(device)
test_out = test_block(test_x)

print("Transformer block output shape:", test_out.shape)

Transformer block output shape: torch.Size([16, 64, 64])


In [6]:
# ============================================================
# BLOCK 6 — Full Tiny GPT Language Model
# ============================================================

class TinyGPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        # Token embedding: token id -> vector
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # Position embedding: position index -> vector
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Transformer blocks
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head) for _ in range(n_layer)
        ])

        # Final normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # Convert embedding vector back to vocab logits
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are [B, T]
        token_emb = self.token_embedding_table(idx)  # [B, T, n_embd]

        pos_ids = torch.arange(T, device=device)
        pos_emb = self.position_embedding_table(pos_ids)  # [T, n_embd]

        x = token_emb + pos_emb  # [B, T, n_embd]

        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)  # [B, T, vocab_size]

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)

            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
        """
        idx: [B, T]
        Generates text autoregressively.
        """
        for _ in range(max_new_tokens):

            # Crop context to block_size
            idx_cond = idx[:, -block_size:]

            # Get predictions
            logits, loss = self(idx_cond)

            # Focus only on last time step
            logits = logits[:, -1, :]  # [B, vocab_size]

            # Temperature controls randomness
            logits = logits / temperature

            # Convert to probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample next token
            idx_next = torch.multinomial(probs, num_samples=1)

            # Append next token
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


# Create the model
model = TinyGPTLanguageModel().to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params:,}")

# Test forward pass
xb, yb = get_batch("train")
logits, loss = model(xb, yb)

print("Logits shape:", logits.shape)
print("Initial loss:", loss.item())

# Generate before training
start_text = "Instruction:"
start_ids = torch.tensor([encode(start_text)], dtype=torch.long, device=device)

generated_ids = model.generate(start_ids, max_new_tokens=300, temperature=1.0)

print("\n--- Generated text before training ---")
print(decode(generated_ids[0].tolist()))

Number of parameters: 108,839
Logits shape: torch.Size([16, 64, 39])
Initial loss: 3.82816219329834

--- Generated text before training ---
Instruction:?cfv,xwrAhkshozshI
AswzATgTfRx:aWcfS,.AkvAmoTTbTf,RnRtgi?kitevEuE lxoHh?Sl,nEwnf?.hy
,.
hu.AWH gtF:Hl,RWRtv?I
fRtmtlgdz?R, t
TRnzvHk aTw khdcfnHtvesazAStWlRvfddf.eEbSfSf:uzdmlbxlvAFsT:tlHSuwE.zHvb
hdzHwShs?HWRaS zAzdpslaTep,rgbeWa vAsdfzz?fu?W,w,pgWWTWAvT:SIis,dHzzWudeTbHnzbsStE
onr,fbrSiEEdv.sbgvfF


In [7]:
# ============================================================
# BLOCK 7 — Training Loop
# ============================================================

@torch.no_grad()
def estimate_loss():
    """
    Calculates average train and validation loss.
    """
    out = {}

    model.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean().item()

    model.train()
    return out


# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training...")

for iter in range(max_iters + 1):

    # Evaluate sometimes
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(
            f"step {iter}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    # Get one batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Training complete.")

Starting training...
step 0: train loss 3.8202, val loss 3.8327
step 200: train loss 1.7601, val loss 2.5259
step 400: train loss 0.9252, val loss 3.1678
step 600: train loss 0.3538, val loss 3.9421
step 800: train loss 0.1722, val loss 4.4862
step 1000: train loss 0.1184, val loss 4.7393
step 1200: train loss 0.1026, val loss 4.8408
step 1400: train loss 0.0887, val loss 5.1818
step 1600: train loss 0.0841, val loss 5.2173
step 1800: train loss 0.0769, val loss 5.5603
step 2000: train loss 0.0753, val loss 5.4302
Training complete.


In [12]:
# ============================================================
# BLOCK 11 — Production-Style Masked Instruction Tuning
# Loss only on Answer tokens
# ============================================================

# Each example has separate instruction and answer.
instruction_examples = [
    {
        "instruction": "Say hello.",
        "answer": "Hello, how can I help you?"
    },
    {
        "instruction": "Explain AI simply.",
        "answer": "AI is software that learns patterns from data."
    },
    {
        "instruction": "What is fine tuning?",
        "answer": "Fine tuning is additional training on a specific task or style."
    },
    {
        "instruction": "What is instruction tuning?",
        "answer": "Instruction tuning means showing the model examples of instructions and answers."
    },
    {
        "instruction": "What is a language model?",
        "answer": "A language model learns patterns from text and predicts what comes next."
    },
    {
        "instruction": "What is the goal of this experiment?",
        "answer": "The goal is to understand tokenization, training, generation, and instruction tuning."
    },
    {
        "instruction": "Is this model powerful?",
        "answer": "no, this model is very small and only useful for learning."
    },
    {
        "instruction": "What does Ravi learn?",
        "answer": "Ravi learns how artificial intelligence models work."
    },
]

# Check all characters exist in the existing tokenizer
all_sft_text = ""

for ex in instruction_examples:
    prompt = f"Instruction: {ex['instruction']}\nAnswer:"
    answer = " " + ex["answer"] + "\n"
    all_sft_text += prompt + answer

missing_chars = sorted(list(set(all_sft_text) - set(stoi.keys())))

print("Missing characters:", missing_chars)
print("Missing count:", len(missing_chars))

if len(missing_chars) > 0:
    raise ValueError(
        f"SFT examples contain characters not in tokenizer: {missing_chars}"
    )


def build_sft_example(instruction, answer):
    """
    Builds one production-style SFT example.

    Input text:
      Instruction: ...
      Answer: ...

    Labels:
      -100 for prompt part
      actual token ids for answer part
    """

    prompt = f"Instruction: {instruction}\nAnswer:"
    answer_text = " " + answer + "\n"

    full_text = prompt + answer_text

    full_ids = encode(full_text)
    prompt_ids = encode(prompt)

    x = torch.tensor(full_ids[:-1], dtype=torch.long)
    y = torch.tensor(full_ids[1:], dtype=torch.long)

    # Mask prompt tokens.
    # y predicts next character, so answer loss starts around len(prompt_ids) - 1.
    labels = y.clone()
    labels[:len(prompt_ids) - 1] = -100

    return x, labels, full_text


sft_dataset = []

for ex in instruction_examples:
    x, labels, full_text = build_sft_example(
        ex["instruction"],
        ex["answer"]
    )

    # Skip examples longer than block_size for this simple toy model
    if len(x) <= block_size:
        sft_dataset.append((x, labels, full_text))
    else:
        print("Skipped long example:")
        print(full_text)
        print("Length:", len(x), "block_size:", block_size)

print("SFT examples kept:", len(sft_dataset))

for i, (x, labels, full_text) in enumerate(sft_dataset[:2]):
    print("=" * 80)
    print("Example", i)
    print(full_text)
    print("Input length:", len(x))
    print("Loss tokens:", (labels != -100).sum().item())

Missing characters: []
Missing count: 0
Skipped long example:
Instruction: Explain AI simply.
Answer: AI is software that learns patterns from data.

Length: 86 block_size: 64
Skipped long example:
Instruction: What is fine tuning?
Answer: Fine tuning is additional training on a specific task or style.

Length: 105 block_size: 64
Skipped long example:
Instruction: What is instruction tuning?
Answer: Instruction tuning means showing the model examples of instructions and answers.

Length: 129 block_size: 64
Skipped long example:
Instruction: What is a language model?
Answer: A language model learns patterns from text and predicts what comes next.

Length: 119 block_size: 64
Skipped long example:
Instruction: What is the goal of this experiment?
Answer: The goal is to understand tokenization, training, generation, and instruction tuning.

Length: 143 block_size: 64
Skipped long example:
Instruction: Is this model powerful?
Answer: no, this model is very small and only useful for learning

In [13]:
# ============================================================
# BLOCK 12 — Train With Production-Style Masked SFT
# Loss only on answer characters
# ============================================================

sft_batch_size = min(4, len(sft_dataset))
sft_learning_rate = 5e-4
sft_max_iters = 800
sft_eval_interval = 100

if len(sft_dataset) == 0:
    raise ValueError(
        "No SFT examples available. Most likely all examples were longer than block_size."
    )

def get_sft_batch():
    """
    Randomly samples SFT examples.

    x      = full prompt + answer input ids
    labels = -100 for prompt chars, real ids for answer chars
    """
    batch_indices = torch.randint(len(sft_dataset), (sft_batch_size,))

    xs = []
    ys = []

    for idx in batch_indices:
        x, labels, _ = sft_dataset[idx]

        # Pad to block_size
        pad_len = block_size - len(x)

        x_padded = torch.cat([
            x,
            torch.zeros(pad_len, dtype=torch.long)
        ])

        y_padded = torch.cat([
            labels,
            torch.full((pad_len,), -100, dtype=torch.long)
        ])

        xs.append(x_padded)
        ys.append(y_padded)

    X = torch.stack(xs).to(device)
    Y = torch.stack(ys).to(device)

    return X, Y


def masked_sft_loss(logits, labels):
    """
    logits shape: [B, T, vocab_size]
    labels shape: [B, T]

    labels contain:
      -100 = ignore this position
      token id = calculate loss
    """
    B, T, C = logits.shape

    logits_flat = logits.view(B * T, C)
    labels_flat = labels.view(B * T)

    loss = F.cross_entropy(
        logits_flat,
        labels_flat,
        ignore_index=-100
    )

    return loss


@torch.no_grad()
def estimate_sft_loss():
    model.eval()

    losses = []

    for _ in range(30):
        X, Y = get_sft_batch()
        logits, _ = model(X)
        loss = masked_sft_loss(logits, Y)
        losses.append(loss.item())

    model.train()

    return sum(losses) / len(losses)


optimizer = torch.optim.AdamW(model.parameters(), lr=sft_learning_rate)

print("Starting masked instruction tuning...")

for step in range(sft_max_iters + 1):

    if step % sft_eval_interval == 0:
        avg_loss = estimate_sft_loss()
        print(f"step {step}: masked SFT loss {avg_loss:.4f}")

    X, Y = get_sft_batch()

    logits, _ = model(X)
    loss = masked_sft_loss(logits, Y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Masked instruction tuning complete.")

Starting masked instruction tuning...
step 0: masked SFT loss 0.0012
step 100: masked SFT loss 0.0000
step 200: masked SFT loss 0.0000
step 300: masked SFT loss 0.0000
step 400: masked SFT loss 0.0000
step 500: masked SFT loss 0.0003
step 600: masked SFT loss 0.0000
step 700: masked SFT loss 0.0000
step 800: masked SFT loss 0.0000
Masked instruction tuning complete.


In [22]:
# ============================================================
# BLOCK 14 — Better Instruction Generation With Stop Character
# Stop after the first generated newline
# ============================================================

@torch.no_grad()
def generate_answer(prompt, max_new_tokens=120, temperature=0.4):
    model.eval()

    input_ids = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        idx_cond = generated[:, -block_size:]

        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        generated = torch.cat([generated, next_id], dim=1)

        next_char = decode([next_id.item()])

        # Stop after model generates newline after answer starts
        if next_char == "\n":
            break

    return decode(generated[0].tolist())


test_prompts = [
    "Instruction: Explain AI simply\nAnswer:",
]

for prompt in test_prompts:
    print("=" * 80)
    print(generate_answer(prompt, max_new_tokens=120, temperature=0.3))

Instruction: Explain AI simply
Answer: how can I help you?

